In [44]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

from google.colab import drive
drive.mount('/content/drive')

data = pd.read_csv(
    '/content/drive/MyDrive/Projects/housing_prediction/Housing.csv'
)

#Features
data["mainroad"] = data["mainroad"].map({"yes": 1, "no": 0})
data["guestroom"] = data["guestroom"].map({"yes": 1, "no": 0})
data["basement"] = data["basement"].map({"yes": 1, "no": 0})
data["hotwaterheating"] = data["hotwaterheating"].map({"yes": 1, "no": 0})
data["airconditioning"] = data["airconditioning"].map({"yes": 1, "no": 0})
data["prefarea"] = data["prefarea"].map({"yes": 1, "no": 0})

data["furnishingstatus"] = data["furnishingstatus"].map(
    {
        "unfurnished": -1,
        "semi-furnished": 0,
        "furnished": 1,
    }
)

X = data.drop(columns=["price"]).values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)

y_log = np.log1p(data["price"].values)

y_mean = y_log.mean()
y_std = y_log.std()

y = (y_log - y_mean) / y_std

#Dataset
class DataFrameDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(
            features,
            dtype=torch.float32
        )
        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

dataset = DataFrameDataset(X, y)

train_size = int(0.75 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

#Model Structure
model = nn.Sequential(
    nn.Linear(X.shape[1], 16),
    nn.ReLU(),
    nn.Dropout(0.15),
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Dropout(0.05),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

criterion = nn.MSELoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

#Training
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=500):

    training_losses = []
    validation_losses = []

    best_val = float("inf")
    patience = 50
    counter = 0

    best_model_state = None

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for inputs, targets in train_loader:
            targets = targets.unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for inputs, targets in val_loader:
                targets = targets.unsqueeze(1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        training_losses.append(avg_train_loss)
        validation_losses.append(avg_val_loss)

        if avg_val_loss < best_val:
            best_val = avg_val_loss
            counter = 0
            best_model_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            counter += 1
        if counter >= patience and epoch >= num_epochs / 20:
            print(f"Early stopping at epoch {epoch+1}")
            break
        if (epoch + 1) % 25 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return training_losses, validation_losses

#Training
training_losses, validation_losses = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=500
)

#Model Evaluation
def evaluate_model(model, loader, y_mean, y_std):

    model.eval()
    preds_list = []
    targets_list = []

    with torch.no_grad():
        for inputs, targets in loader:
            targets = targets.unsqueeze(1)
            preds = model(inputs)
            preds_list.append(preds)
            targets_list.append(targets)

    preds = torch.cat(preds_list)
    targets = torch.cat(targets_list)
    preds_log = preds * y_std + y_mean
    targets_log = targets * y_std + y_mean
    preds_price = torch.expm1(preds_log)
    targets_price = torch.expm1(targets_log)

    rmse = torch.sqrt(
        torch.mean(
            (preds_price - targets_price) ** 2
        )
    ).item()
    mae = torch.mean(
        torch.abs(
            preds_price - targets_price
        )
    ).item()
    r2 = r2_score(
        targets_price.numpy(),
        preds_price.numpy()
    )

    return rmse, mae, r2

#Results/Metrics
train_rmse, train_mae, train_r2 = evaluate_model(
    model,
    train_loader,
    y_mean,
    y_std
)
val_rmse, val_mae, val_r2 = evaluate_model(
    model,
    val_loader,
    y_mean,
    y_std
)
test_rmse, test_mae, test_r2 = evaluate_model(
    model,
    test_loader,
    y_mean,
    y_std
)

print("\n===== FINAL RESULTS =====")
print(
    f"Train RMSE: ${train_rmse:,.2f} | "
    f"Train MAE: ${train_mae:,.2f} | "
    f"Train R²: {train_r2:.4f}"
)
print(
    f"Val RMSE: ${val_rmse:,.2f} | "
    f"Val MAE: ${val_mae:,.2f} | "
    f"Val R²: {val_r2:.4f}"
)
print(
    f"Test RMSE: ${test_rmse:,.2f} | "
    f"Test MAE: ${test_mae:,.2f} | "
    f"Test R²: {test_r2:.4f}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 25/500 | Train Loss: 0.9957 | Val Loss: 0.7151
Epoch 50/500 | Train Loss: 0.7238 | Val Loss: 0.5324
Epoch 75/500 | Train Loss: 0.4582 | Val Loss: 0.3261
Epoch 100/500 | Train Loss: 0.3568 | Val Loss: 0.2903
Epoch 125/500 | Train Loss: 0.3587 | Val Loss: 0.2924
Epoch 150/500 | Train Loss: 0.3314 | Val Loss: 0.2953
Early stopping at epoch 160

===== FINAL RESULTS =====
Train RMSE: $1,098,282.12 | Train MAE: $771,933.62 | Train R²: 0.6788
Val RMSE: $1,064,290.38 | Val MAE: $769,937.62 | Val R²: 0.6322
Test RMSE: $804,675.75 | Test MAE: $605,361.38 | Test R²: 0.7004
